In [1]:
import langchain
print(langchain.__version__)

1.3.18


In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(".env")
gemini_key_exists = bool(os.getenv("GEMINI_API_KEY"))
print(gemini_key_exists)

In [4]:
%pip install -q langchain-google-genai


Note: you may need to restart the kernel to use updated packages.


In [30]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash",
    temperature=0.7,
    max_tokens=None,
    timeout=None,
    max_retries=2,)

In [6]:
llm_response = llm.invoke("Describe the Artemis 3 mission in 50 words")

print(llm_response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Artemis III is NASA’s historic mission to land humans on the Moon for the first time in over fifty years. Launching aboard the SLS rocket, astronauts will journey in Orion and transfer to SpaceX’s Starship to land at the lunar South Pole, exploring for water ice and preparing for Mars.


In [7]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()
output_parser.invoke(llm_response)

'Artemis III is NASA’s historic mission to land humans on the Moon for the first time in over fifty years. Launching aboard the SLS rocket, astronauts will journey in Orion and transfer to SpaceX’s Starship to land at the lunar South Pole, exploring for water ice and preparing for Mars.'

In [8]:
chain = llm | output_parser

In [9]:
from typing import List,Dict
from pydantic import BaseModel, Field


In [10]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("Tell me a joke about {topic}")
prompt.invoke({"topic" : "programmer"})


ChatPromptValue(messages=[HumanMessage(content='Tell me a joke about programmer', additional_kwargs={}, response_metadata={})])

In [11]:
chain = prompt | llm | output_parser
print(chain.invoke({"topic" : "performance pressure"}))

A man goes to a therapist to cope with the extreme performance pressure he feels at work. 

The therapist smiles warmly and says, "Don't worry, it’s all in your head. Whenever you feel overwhelmed, just take a deep breath and tell yourself: *'It’s not life or death. If I mess up, nobody dies.'*"

The man looks at him in sheer panic and says, "But Doctor... I'm a heart surgeon!"


In [12]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from typing import List
from langchain_core.documents import Document

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 200,
    length_function = len
)

docx_loader = Docx2txtLoader("docs/GreenGrow Innovations_ Company History.docx")
# Install the required dependency for Docx2txtLoader
# docx_loader += Docx2txtLoader()
# Load the document
document = docx_loader.load()

print(len(document))

splits = text_splitter.split_documents(document)

print(f"Split the documents into {len(splits)} chunks.")


1
Split the documents into 6 chunks.


C:\Users\Avinash\AppData\Local\Temp\ipykernel_9300\2328834074.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader


In [13]:
print(splits[0])

page_content='GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient.' metadata={'source': 'docs/GreenGrow Innovations_ Company History.docx'}


In [14]:
def load_documents(folder_path:str) -> List[Document]:
    
    documents = []
    
    for filename in os.listdir(folder_path):
        loader:str
        file_path = os.path.join(folder_path, filename)
        
        if filename.endswith('.pdf'):
            loader = PyPDFLoader(file_path)
        elif filename.endswith('.docx'):
            loader = Docx2txtLoader(file_path)
        else:
            print(f"Unsupported file type : {filename}") 
            continue
        documents.extend(loader.load())

    return documents

    
folder_path = "docs"

documents = load_documents(folder_path)

print(f"Loaded {len(documents)} documents from the folder.")

splits = text_splitter.split_documents(documents)

print(f"Split the documents into {len(splits)} chunks.")

Loaded 5 documents from the folder.
Split the documents into 19 chunks.


In [15]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

document_embeddings = embeddings.embed_documents([split.page_content for split in splits])

print(f"Created embeddings for {len(document_embeddings)} dcoument chunks")

Created embeddings for 19 dcoument chunks


In [16]:
from langchain_chroma import Chroma

embedding_function = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")
collection_name = "my_collection"
vectorstore = Chroma.from_documents(collection_name=collection_name, documents=splits, embedding=embedding_function)

# print("Vec")

In [17]:
from langchain_core.prompts import ChatPromptTemplate

template = """ Answer the question based only on the following context:
{context}

Question: {question}

Answer:
"""
prompt = ChatPromptTemplate.from_template(template)

In [18]:
#  = vectorstore.as_retriever(search_kwargs={k:2})
retriever = vectorstore.as_retriever(search_kwargs={"k":2})
retriever.invoke("When was Greengrow inov founded")

[Document(id='4d65091a-9ecf-4140-a069-f8e80f85f7bc', metadata={'source': 'docs\\GreenGrow Innovations_ Company History.docx'}, page_content='GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient.'),
 Document(id='2496f82a-66e7-4f52-ba45-1df6192f6cf0', metadata={'source': 'docs\\GreenGrow Innovations_ Company History.docx'}, page_content='Today, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in vertical farming, drought-resistant crop development, and AI-powered farm management systems.')]

In [19]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever,
     "question": RunnablePassthrough()}
    | prompt
)

rag_chain.invoke("When was Greengrow inov founded")


ChatPromptValue(messages=[HumanMessage(content=" Answer the question based only on the following context:\n[Document(id='4d65091a-9ecf-4140-a069-f8e80f85f7bc', metadata={'source': 'docs\\\\GreenGrow Innovations_ Company History.docx'}, page_content='GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient.'), Document(id='2496f82a-66e7-4f52-ba45-1df6192f6cf0', metadata={'source': 'docs\\\\GreenGrow Innovations_ Company History.docx'}, page_content='Today, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in vertical farming, drought-resistant crop development, and AI-powered farm manag

In [20]:
def doc2str(docs) :
    return "\n\n".join(doc.page_content for doc in docs)

In [21]:
# rag_chain = (
#     {"content": retriever | doc2str, "question": RunnablePassthrough()} | prompt
# )

rag_chain = (
    {"context": retriever | doc2str, "question": RunnablePassthrough()}
    | prompt
)

rag_chain.invoke("When was Greengrow inov founded")

ChatPromptValue(messages=[HumanMessage(content=' Answer the question based only on the following context:\nGreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient.\n\nToday, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in vertical farming, drought-resistant crop development, and AI-powered farm management systems.\n\nQuestion: When was Greengrow inov founded\n\nAnswer:\n', additional_kwargs={}, response_metadata={})])

In [22]:

rag_chain = (
    {"context": retriever | doc2str, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
question = "When was GreenGrow Innovations founded?"
response = rag_chain.invoke(question)
print(response)

Based on the provided context, GreenGrow Innovations was founded in 2010.


In [23]:
# Example conversation
from langchain_core.messages import HumanMessage, AIMessage
chat_history = []
chat_history.extend([
    HumanMessage(content=question),
    AIMessage(content=response)
])

In [24]:
from langchain_core.prompts import MessagesPlaceholder
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# history_aware_retriever = create_history_aware_retriever(
#     llm, retriever, contextualize_q_prompt
# )
contextualize_chain = contextualize_q_prompt | llm | StrOutputParser()
contextualize_chain.invoke({"input": "Where it is headquartered?", "chat_history": chat_history})

'Where is GreenGrow Innovations headquartered?'

In [25]:
test = embeddings.embed_query("Hello world")
print(len(test))

3072


In [26]:
print(embeddings.embed_query("Where is it headquartered?"))

[0.006832535, 0.03025004, -0.013958414, 0.00071773987, 0.0049529583, 0.006543146, 0.006119984, -0.01774223, -0.00050746044, -0.045788266, 0.0025632554, -0.0045123226, -0.00018788244, -0.027175063, -0.00041619418, 0.002898482, -0.0025739323, -0.0059059616, 0.00018638148, -0.023340102, -0.023419186, 0.005726697, 0.036311876, 0.0019775883, -0.02038242, 0.017623391, 0.0054318192, -0.020631162, -0.012265367, 0.12977816, -0.040584847, 0.0008840259, -0.002462353, -0.020540552, -0.009222475, -0.003619326, 0.00032060294, -0.008681205, 0.02206805, -0.008557798, 0.0024625158, 0.01903857, 0.00066899334, 0.0051464993, -0.0017265893, -0.00987604, 0.012814932, -0.034921385, 0.03699809, -0.007918172, 0.012494431, 0.010925493, 0.012917979, -0.018550072, 0.0066597727, 0.0014531966, -0.012573358, 0.026727144, 0.0009491711, 0.0010312367, 0.014196893, 0.03215423, -0.0037380152, -0.0029861168, -0.01313421, -0.00021056483, 0.004932213, 0.0018657253, 0.027501073, 0.0018184088, -0.019680355, 0.0032515787, -0.0

In [27]:
# retriever = vectorstore.as_retriever() 

from langchain_classic.chains import create_history_aware_retriever

history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
history_aware_retriever.invoke({"input": "Where it is headquartered?", "chat_history": chat_history})

[Document(id='2496f82a-66e7-4f52-ba45-1df6192f6cf0', metadata={'source': 'docs\\GreenGrow Innovations_ Company History.docx'}, page_content='Today, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in vertical farming, drought-resistant crop development, and AI-powered farm management systems.'),
 Document(id='4d65091a-9ecf-4140-a069-f8e80f85f7bc', metadata={'source': 'docs\\GreenGrow Innovations_ Company History.docx'}, page_content='GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient.')]

In [28]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

qa_prompt = ChatPromptTemplate.from_messages(
            [
                ( "system" , "You are a helpful AI assistant. Use the following context to answer the user's question.") ,
                ( "system" , "Context: {context}"),
                ( "human" , "{input}")
            ]
)

question_answer_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)

rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [31]:
rag_chain.invoke({"input": "Where is it headquartered?", "chat_history": chat_history})

{'input': 'Where is it headquartered?',
 'chat_history': [HumanMessage(content='When was GreenGrow Innovations founded?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Based on the provided context, GreenGrow Innovations was founded in 2010.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'context': [Document(id='2496f82a-66e7-4f52-ba45-1df6192f6cf0', metadata={'source': 'docs\\GreenGrow Innovations_ Company History.docx'}, page_content='Today, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in vertical farming, drought-resistant crop development, and AI-powered farm management systems.'),
  Document(id='4d65091a-9ecf-4140-a069-f8e80f85f7bc', metadata={'source': 'docs\\GreenGrow Innovations_ Company History.docx'}, page_content='GreenGrow Innovations w

In [32]:
import sqlite3
from datetime import datetime

DB_name = "rag_app.db"
def get_db_connection():
    conn = sqlite3.connect(DB_name)
    conn.row_factory = sqlite3.Row
    return conn

In [ ]:

def create_application_logs():
  conn = get_db_connection()
  conn.execute("""
    CREATE TABLE IF NOT EXISTS application_logs (
      id INTEGER PRIMARY KEY AUTOINCREMENT,
      session_id TEXT,
      user_query TEXT,
      gemini_response TEXT,
      model TEXT,
      created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
  """)
  conn.commit()
  conn.close()


def insert_application_logs(session_id, user_query, gemini_response, model):
  conn = get_db_connection()
  conn.execute(
    """
    INSERT INTO application_logs
    (session_id, user_query, gemini_response, model)
    VALUES (?, ?, ?, ?)
    """,
    (session_id, user_query, gemini_response, model)
  )
  conn.commit()
  conn.close()


def get_chat_history(session_id):
  conn = get_db_connection()
  cursor = conn.cursor()
  cursor.execute(
    """
    SELECT user_query, gemini_response
    FROM application_logs
    WHERE session_id = ?
    ORDER BY created_at
    """,
    (session_id,)
  )

  messages = []
  for row in cursor.fetchall():
    messages.extend([
      {"role": "human", "content": row["user_query"]},
      {"role": "ai", "content": row["gemini_response"]},
    ])

  conn.close()
  return messages


create_application_logs()

In [46]:
import uuid

session_id = str(uuid.uuid4())
chat_history = get_chat_history(session_id)
print(chat_history)
question1 = "When was GreenGrow Inov founded?"
answer1 = rag_chain.invoke({"input": question1, "chat_history": chat_history})['answer']

insert_application_logs(session_id, question1, answer1, "gemini-3.1-flash")

print(f'Human: {question1}')
print(f'AI: {answer1}')

[]
Human: When was GreenGrow Inov founded?
AI: GreenGrow Innovations was founded in **2010**.


In [50]:
print(session_id)

6c4c2170-446c-40e0-a5fb-938ca7685b7e


In [55]:
question2 = "Where is it headquartered?"
chat_history = get_chat_history(session_id)
print(chat_history)

answer2 = rag_chain.invoke({"input" : question2, "chat_history" : chat_history})['answer']
insert_application_logs(session_id, question2, answer2, "gemini-3.1-flash")
print(f'Human: {question2}')
print(f'AI: {answer2}')

[{'role': 'human', 'content': 'When was GreenGrow Inov founded?'}, {'role': 'ai', 'content': 'GreenGrow Innovations was founded in **2010**.'}, {'role': 'human', 'content': 'Where is it headquartered?'}, {'role': 'ai', 'content': 'Based on the provided context, it is not explicitly stated where GreenGrow Innovations is currently headquartered. However, the company started in **Portland, Oregon**, and has since expanded its operations to include offices in **California** and **Iowa**.'}]
Human: Where is it headquartered?
AI: Based on the provided context, it is not explicitly stated where GreenGrow Innovations is currently headquartered. However, the company originally started in **Portland, Oregon**, and has since expanded its operations to include offices in **California** and **Iowa**.
